# Final model: 2x2D MaxViT + 3D ResNeXt + CrossGate (Bilateral 200^3)

Canonical training notebook for the 3-branch model on the **Bilateral-denoised** dataset at raw **200^3**
(`STORE_RES=RES3D=200`, 2D views cached at 224 from the denoised volume). Architecture lives in
`scripts/final_model.py`; recovery, caching, metrics, reporting and X-AI live in `scripts/final_training.py`.
Follows `docs/notebook-conventions.md`.

**Recovery contract:** `last.pt` is committed only after a complete optimizer/scaler/scheduler operation with cleared gradients, periodically and every epoch. It includes the model, optimizer, scheduler, scaler, RNG, deterministic sample order/cursor, partial epoch metrics, history and best state. Resume replays work since the last successful commit, not an arbitrary interrupt point. Per-sample augmentation depends only on seed/epoch/index, with zero loader workers. Same software/device/data are required for numerical reproducibility; CUDA bitwise equivalence is not promised. W&B may contain repeated progress values for replayed work.

First SIGINT requests a stop at the next safe boundary. A second interrupt or other exception saves separate **non-resumable emergency weights**, which may reflect a partial optimizer operation; it never intentionally replaces `last.pt`. Local commits survive Drive copy failures, which raise an explicit error and leave a `.sync-pending.pt` record. Re-run with `RESUME=True` after fixing storage. `best.pt` is a full checkpoint at an improved validation AUC; `last.pt` also contains the best weights.

**Current preset: full Bilateral training at 200^3.** Raw 200^3 volumes are downloaded from HF with `HF_TOKEN`
(`allow_patterns` for the declared splits only), denoised once with the parameter-tracked Bilateral method
(resume-safe partial caches + completion markers), and all three splits and both 2D views train on the denoised
volumes. Optimizer and scheduler start fresh; set both `WARM_START_WEIGHTS`/`WARM_START_TARGET` to fine-tune from
a checkpoint instead (e.g. the recovered raw model). Metrics follow the conventions: train per optimizer step,
val + test per epoch, calibrated `train/val/test` + bootstrap CI + `report/split_table` in W&B. `RUN_XAI=True`
explains the best model (Grad-CAM 3D/2D, occlusion, integrated gradients, fusion attention, branch drop) and logs
`xai/fusion_table` + `xai/*`.

`FINAL_SMOKE=1` uses tiny synthetic CPU data and a separate tiny model, no dataset/pretrained downloads. W&B
offline is explicitly permitted only for this local smoke verification; it does not test online W&B or Drive.
Setup installs dependencies only for real Colab execution. Editing this notebook does not alter an already-running
external kernel.

In [ ]:
import os, sys, subprocess
from pathlib import Path
SMOKE = os.environ.get('FINAL_SMOKE', '0') == '1'
if not SMOKE and 'google.colab' in sys.modules:
    repo = Path('/content/glaucoma-thesis')
    if not repo.exists():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Tqhuyen/glaucoma-thesis.git', str(repo)], check=True)
    os.chdir(repo)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm', 'wandb', 'scikit-image', 'scipy', 'matplotlib', 'huggingface_hub', 'python-dotenv', 'hf-transfer'], check=True)
    os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER', '1')
sys.path.insert(0, str(Path.cwd()))
import csv, json, random, tempfile, time
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
from scripts import final_model as fm, final_training as ft
ft.load_env_file()
DEVICE = torch.device('cpu' if SMOKE else ('cuda' if torch.cuda.is_available() else 'cpu'))
if DEVICE.type == 'cuda':
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
if SMOKE:
    torch.set_num_threads(1)


## Configuration
Use a distinct `RUN_GROUP` for a new experiment. `RESUME=True` requires matching config, data identities and a
saved W&B id. `RUN_TARGET` blank selects all datasets/seeds. `STORE_RES`/`RES3D`/`RES2D` are config-driven; this
preset stores and trains the 3D branch at **200^3** with 2D views at 224 projected from the denoised volume.
`DENOISE_LIMIT` is the maximum additional samples processed per split per invocation; incomplete caches cannot
enter training. Set both `WARM_START_WEIGHTS` and `WARM_START_TARGET` to fine-tune from a checkpoint.

Model init order: resume checkpoint (`last.pt`) > existing weights at `WARM_START_WEIGHTS`/`best_weights.pt`
(loaded into the model, pretrained backbone skipped) > only when no weights exist is a new model initialized
with a pretrained 2D backbone.

Training extension: to train N more epochs after a finished or interrupted run, set `RESUME=True` and
`EXTEND_EPOCHS=N`, then re-run the training cell. The run keeps its identity (other config must match; `epochs`
is the only allowed difference), the identity and `last.pt` target are updated, `completed.pt` is reopened,
early-stop patience is reset and the LR follows the new total cosine schedule.

In [ ]:
RUN_GROUP = 'bilateral_s42_full'
RUN_TARGET = 'bilateral_s42'
RESUME = False
EXTEND_EPOCHS = 0
WARM_START_WEIGHTS = ''
WARM_START_TARGET = ''
CHECKPOINT_EVERY_STEPS = 10
DATASETS = ['bilateral']
SEEDS = [42]
STORE_RES = 8 if SMOKE else 200
RES3D = 8 if SMOKE else 200
RES2D = 8 if SMOKE else 224
N_2D, D_LATENT, ENC2D = 2, 256, 'maxvit_tiny_rw_224'
EPOCHS, BS, GRAD_ACCUM = (1, 2, 2) if SMOKE else (10, 2, 8)
LR, WD, PATIENCE = 1e-4, 1e-4, 4
BUILD_DENOISED, DENOISE_METHOD, DENOISE_LIMIT = True, 'bilateral', 0
RUN_XAI = True
DENOISE_PARAMS = {'sigma_color': 0.10, 'sigma_spatial': 4.0}
DENOISE_IMPLEMENTATION = None
HF_RAW_REPO = os.environ.get('HF_DATA_REPO', 'tqhuyen/harvard-oct-glaucoma-200')
SPLITS = ('Training', 'Validation', 'Test')
HF_RAW_PATTERNS = [f'{split}_{kind}.npy' for split in SPLITS for kind in ('volumes', 'labels')]
SMOKE_ROOT = Path(tempfile.mkdtemp(prefix='final_smoke_')) if SMOKE else None
DATA_ROOT = SMOKE_ROOT / 'data' if SMOKE else Path('/content/final_data')
LOCAL_ROOT = SMOKE_ROOT / 'runs' if SMOKE else Path('outputs/final_2x2d_3d_crossgate') / RUN_GROUP
DRIVE_MOUNT = Path(os.environ.get('DRIVE_MOUNT', '/content/drive'))
DRIVE_ROOT = Path(os.environ.get('DRIVE_ROOT', '/content/drive/MyDrive/MasterBKDN/Thesis'))
DRIVE_DIR = DRIVE_ROOT / 'final_2x2d_3d_crossgate' / RUN_GROUP
selected = [(ds, seed, f'{ds}_s{seed}') for ds in DATASETS for seed in SEEDS if not RUN_TARGET or RUN_TARGET == f'{ds}_s{seed}']
if not selected:
    raise ValueError('RUN_TARGET does not match DATASETS/SEEDS')
if bool(WARM_START_WEIGHTS) != bool(WARM_START_TARGET):
    raise ValueError('Set both WARM_START_WEIGHTS and WARM_START_TARGET')
if WARM_START_WEIGHTS and (RESUME or RUN_TARGET != WARM_START_TARGET or len(selected) != 1):
    raise ValueError('Warm-start requires RESUME=False and one explicitly selected matching RUN_TARGET')
if not SMOKE:
    if not os.path.ismount(DRIVE_MOUNT):
        from google.colab import drive
        drive.mount(str(DRIVE_MOUNT))
    if not os.path.ismount(DRIVE_MOUNT) or not DRIVE_ROOT.resolve().is_relative_to(DRIVE_MOUNT.resolve()):
        raise RuntimeError('Drive must be a verified mounted filesystem, not a local directory')
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
if WARM_START_WEIGHTS and not Path(WARM_START_WEIGHTS).is_file():
    raise FileNotFoundError(WARM_START_WEIGHTS)
DATA_ROOT.mkdir(parents=True, exist_ok=True)
STORAGE = ft.Artifacts(LOCAL_ROOT, None if SMOKE else DRIVE_DIR, smoke=SMOKE)
DATA_STORAGE = ft.Artifacts(DATA_ROOT, None if SMOKE else DRIVE_DIR / 'data', smoke=SMOKE)
print('Device:', DEVICE, '| storage', STORE_RES, '| res3d', RES3D, '| res2d', RES2D, '| xai', RUN_XAI)


## Data and caches
Raw 200^3 volumes are downloaded from HF **with credentials** (`HF_TOKEN`, `allow_patterns` for the declared
splits only) and denoised once with the parameter-tracked Bilateral method; completed denoised arrays are cached
and reused, and are worth considering for a versioned HF upload if future runs reuse them. Both 4D `(N,D,H,W)`
and 5D `(N,1,D,H,W)` storage are indexed per sample. Denoising uses partial files plus a committed cursor and
completion marker; legacy unmarked outputs are rebuilt rather than trusted. Views are keyed by their actual
raw/denoised source and resolution and published only when complete. Cache identity uses path/size/mtime;
training config additionally hashes complete volume/label files (cached SHA256). Identical raw data can resume
after re-download. Do not edit files in place while preserving timestamps. Denoise partial caches are local
recovery only; completed denoised arrays and generated views are synced before training.

In [ ]:
if SMOKE:
    rng = np.random.default_rng(0)
    for split, n in zip(SPLITS, (10, 6, 6)):
        np.save(DATA_ROOT / f'{split}_volumes.npy', rng.integers(0, 255, (n, 1, STORE_RES, STORE_RES, STORE_RES), dtype=np.uint8))
        np.save(DATA_ROOT / f'{split}_labels.npy', np.arange(n, dtype=np.int64) % 2)
else:
    from huggingface_hub import snapshot_download
    token = os.environ.get('HF_TOKEN')
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get('HF_TOKEN')
        except Exception:
            token = None
    if not token:
        raise RuntimeError('Authenticated HF download requires HF_TOKEN in .env/environment or Colab Secrets')
    if not all((DATA_ROOT / name).is_file() for name in HF_RAW_PATTERNS):
        snapshot_download(repo_id=HF_RAW_REPO, repo_type='dataset', local_dir=str(DATA_ROOT), token=token, allow_patterns=HF_RAW_PATTERNS)
if any(ds != 'raw' for ds, _, _ in selected):
    from scripts import compare_denoise_methods as cdm
    import skimage, scipy, hashlib
    if DENOISE_METHOD in ('bm3d', 'dncnn', 'swinir'):
        raise ValueError('This parameter-tracked path supports classical fixed-parameter denoisers only')
    if set(DENOISE_PARAMS) != set(cdm.METHODS[DENOISE_METHOD]['params']):
        raise ValueError('DENOISE_PARAMS must specify all and only parameters for the selected method')
    cdm.METHODS[DENOISE_METHOD]['params'] = dict(DENOISE_PARAMS)
    DENOISE_IMPLEMENTATION = {'module_sha256': hashlib.sha256(Path(cdm.__file__).read_bytes()).hexdigest(), 'skimage': skimage.__version__, 'scipy': scipy.__version__, 'numpy': np.__version__}
    if any(ds not in ('raw', DENOISE_METHOD) for ds, _, _ in selected):
        raise ValueError('Dataset tag must match DENOISE_METHOD')
    if BUILD_DENOISED:
        from scripts import compare_denoise_methods as cdm
        def denoise(volume):
            return cdm.denoise_volume(volume, DENOISE_METHOD, workers=2, cache_path=None)[0]
        complete = [ft.build_denoised(DATA_ROOT / f'{s}_volumes.npy', DATA_ROOT / f'{s}_volumes_dn.npy', denoise, method=DENOISE_METHOD, params=DENOISE_PARAMS, implementation=DENOISE_IMPLEMENTATION, limit=DENOISE_LIMIT) for s in SPLITS]
        if not all(complete):
            raise RuntimeError('Denoise incomplete; rerun data cell to continue. Training has not started.')
    for split in SPLITS:
        source = DATA_ROOT / f'{split}_volumes_dn.npy'
        if not source.with_suffix('.complete.pt').exists():
            raise RuntimeError('Denoised cache is not verified complete')
        marker = torch.load(source.with_suffix('.complete.pt'), weights_only=False)
        if marker.get('method') != DENOISE_METHOD or marker.get('params') != DENOISE_PARAMS or marker.get('implementation') != DENOISE_IMPLEMENTATION:
            raise ValueError('Completed denoise cache method/parameters/implementation mismatch')
        DATA_STORAGE.sync(source)
        DATA_STORAGE.sync(source.with_suffix('.complete.pt'))
def make_datasets(ds, seed):
    suffix = 'volumes' if ds == 'raw' else 'volumes_dn'
    datasets = [ft.FinalDataset(DATA_ROOT / f'{s}_{suffix}.npy', DATA_ROOT / f'{s}_labels.npy', res3d=RES3D, res2d=RES2D, seed=seed, train=s == 'Training') for s in SPLITS]
    if not SMOKE and any(tuple(d.volumes.shape[-3:]) != (STORE_RES, STORE_RES, STORE_RES) for d in datasets):
        raise ValueError(f'Real training requires STORE_RES-cubed storage, got {STORE_RES}')
    if ds == 'bilateral' and any(d.source.name != f'{s}_volumes_dn.npy' for s, d in zip(SPLITS, datasets)):
        raise ValueError('Bilateral train/validation/test must all read denoised volumes')
    for split, dataset in zip(SPLITS, datasets):
        print(f'[input] {ds} {split}: {dataset.source} | res3d={RES3D} | views derived from this source at {RES2D}px')
    for split in SPLITS:
        for path in DATA_ROOT.glob(f'{split}_{suffix}_*{RES2D}*'):
            if '.partial.' not in path.name:
                DATA_STORAGE.sync(path)
    return datasets


## Train, evaluate, persist and explain
Metrics cadence: every optimizer step logs the full metric set on the current accumulation window under
`train/*` (plus `train/loss`, running `train/acc`, `train/lr`, `progress/step`); validation and test log the
full set every epoch under `val/*` and `test/*`. After training, `calibrated_report` adds calibrated
`train/val/test` metrics plus bootstrap CI and `log_report` writes the `report/split_table` W&B table.
`RUN_XAI=True` then explains the best model and logs `xai/fusion_table` + `xai/*`. `ACTIVE_TRAINER` and
`ACTIVE_MODEL` remain accessible if training raises; all errors propagate after best-effort emergency
persistence and W&B finish. Calibration is fit on validation logits first, then the threshold is selected on
calibrated validation probabilities and passed to both test metrics and bootstrap CI. Every saved report/XAI
artifact is synced immediately. XAI uses the first validation case, not a test-selected case.

In [ ]:
RESULTS = {}
ACTIVE_MODEL = ACTIVE_TRAINER = WANDB_RUN = None
for ds, seed, tag in selected:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    tr, va, te = make_datasets(ds, seed)
    ytr = tr.labels
    if set(np.unique(ytr)) != {0, 1}:
        raise ValueError('Training requires both binary classes')
    weights = [len(ytr) / (2 * int((ytr == c).sum())) for c in (0, 1)]
    config = dict(dataset=ds, seed=seed, epochs=EPOCHS, batch_size=BS, grad_accum=GRAD_ACCUM, lr=LR, weight_decay=WD, patience=PATIENCE, checkpoint_steps=CHECKPOINT_EVERY_STEPS, class_weights=weights, res3d=RES3D, res2d=RES2D, store_res=STORE_RES, n2d=N_2D, latent=D_LATENT, enc2d=ENC2D, smoke=SMOKE, torch_version=str(torch.__version__), device=str(DEVICE), denoise_method=DENOISE_METHOD if ds != 'raw' else 'none', data=[ft.data_identity(p) for d in (tr, va, te) for p in (d.source, d.label_path)])
    artifacts = ft.Artifacts(LOCAL_ROOT / tag, None if SMOKE else DRIVE_DIR / tag, smoke=SMOKE)
    config.update(denoise_params=DENOISE_PARAMS if ds != 'raw' else {}, denoise_implementation=DENOISE_IMPLEMENTATION if ds != 'raw' else None)
    config.update(run_xai=RUN_XAI, training_phase='bilateral_full')
    status = ft.run_status(artifacts, config, resume=RESUME, extend_epochs=EXTEND_EPOCHS)
    config = status['config']
    print(tag, status['status'])
    if status['status'] == 'complete':
        RESULTS[tag] = {'res': status['result']}
        continue
    tag_resume = status['status'] == 'resume'
    tag_warm_start = status['warm_start'] if status['status'] == 'initialized' else (WARM_START_WEIGHTS if tag == WARM_START_TARGET else '')
    if tag_warm_start and not Path(tag_warm_start).is_file():
        raise FileNotFoundError('Uncommitted warm-start requires its original weights: ' + tag_warm_start)
    WANDB_RUN = ft.init_wandb('final_' + tag, config, artifacts, resume=status['status'] != 'new', smoke=SMOKE, warm_start=tag_warm_start)
    exit_code, stopped = 1, False
    started = time.time()
    try:
        model_path = Path(tag_warm_start) if tag_warm_start else artifacts.local / 'best_weights.pt'
        if tag_resume or not model_path.is_file():
            model_path = None
        ACTIVE_MODEL = (ft.SmokeModel() if SMOKE else fm.FinalModel(n_2d=N_2D, D=D_LATENT, enc2d=ENC2D, enc2d_pretrained=not (tag_resume or bool(tag_warm_start) or model_path is not None))).to(DEVICE)
        if model_path is not None:
            ft.load_weights(ACTIVE_MODEL, model_path)
            print('Loaded existing weights:', model_path)
        else:
            print('No existing weights; initialized a new model')
        def evaluate(model):
            p, y, logits = ft.predict(model, va, BS)
            return {**fm.full_metrics(p, y), 'loss': float(torch.nn.functional.cross_entropy(torch.tensor(logits), torch.tensor(y)))}
        def evaluate_test(model):
            p, y, logits = ft.predict(model, te, BS)
            return {**fm.full_metrics(p, y), 'loss': float(torch.nn.functional.cross_entropy(torch.tensor(logits), torch.tensor(y)))}
        ACTIVE_TRAINER = ft.Trainer(ACTIVE_MODEL, tr, config, artifacts, WANDB_RUN, resume=tag_resume, warm_start=tag_warm_start)
        stopped = not ACTIVE_TRAINER.fit(evaluate, test_evaluate=evaluate_test)
        if stopped:
            WANDB_RUN.summary['stopped_safely'] = True
            exit_code = 0
        else:
            ACTIVE_MODEL.load_state_dict(ACTIVE_TRAINER.best_state)
            res, probs, labels = ft.calibrated_report(ACTIVE_MODEL, va, te, BS, smoke=SMOKE, train=tr)
            res.update(tag=tag, seed=seed, hist=ACTIVE_TRAINER.history, minutes=round((time.time() - started) / 60, 2))
            ft.log_report(WANDB_RUN, res)
            params = sum(p.numel() for p in ACTIVE_MODEL.parameters())
            WANDB_RUN.summary.update({'threshold': res['threshold'], 'temperature': res['temperature'], 'params': params, 'minutes': res['minutes']})
            weights_path = artifacts.save(ft.cpu_state(ACTIVE_MODEL), 'best_weights.pt')
            ft.save_report(res, probs, labels, artifacts, WANDB_RUN)
            RESULTS[tag] = dict(res=res, weights_path=str(weights_path), params=params, test_probs=probs.tolist(), test_labels=labels.tolist())
            if RUN_XAI:
                ft.save_xai(ACTIVE_MODEL, va, artifacts, WANDB_RUN, smoke=SMOKE)
            exit_code = 0
    finally:
        WANDB_RUN.finish(exit_code=exit_code)
    if stopped:
        print('Stopped safely. Set RESUME=True and RUN_TARGET to', tag)
        break
    ft.complete_run(artifacts, config, WANDB_RUN.id)
    ACTIVE_MODEL.cpu()
    ACTIVE_TRAINER = None
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
print('Completed:', list(RESULTS))


## Outputs and limitations
- `metrics.json` / `test_predictions.pt`: calibrated `train/val/test` metrics, bootstrap CI, full history.
- W&B (tables + numbers, no metric plots): `report/split_table`, `report/history_table`, per-step `train/*`,
  per-epoch `val/*` + `test/*`, calibrated `train|val|test/*` with CI, and `xai/fusion_table` + `xai/*`.
- Drive: `MasterBKDN/Thesis/final_2x2d_3d_crossgate/<RUN_GROUP>/<tag>/` (plus `data/` caches).
- Trains on Bilateral-denoised 200^3 volumes (views from the denoised source); switching denoiser or storage
  requires matching `DENOISE_METHOD`/`DENOISE_PARAMS`/`STORE_RES` and a new `RUN_GROUP`.
- Single-seed by default; run 3-5 seeds for mean +/- std. Resume only replays work since the last committed
  optimizer step.

In [ ]:
rows = ft.publish_summary(STORAGE)
if rows:
    print(rows)
else:
    print('No completed runs; no summary or XAI selection attempted.')
print('Smoke verified only local/offline behavior.' if SMOKE else 'Completed artifacts were synced to the verified Drive mount.')
